# Oil Supply Chain Stress Network Analysis
## Feb 15, 2026 (pre-conflict) → Mar 6, 2026 (Hormuz closure)

> **Physics framing:** The supply network is a directed weighted graph G(V,E,w).  
> Node stress is a scalar field on V; edge weights encode dependency strength.  
> The Feb→Mar interval is a **topological phase transition**: distributed stress → hub-collapse regime.

---
**Sources:** IEA Oil Market Report Feb 2026 · EIA STEO Feb 2026 · Kpler tanker analytics · Wikipedia 2026 Hormuz Crisis

### Stress Index Model
```
node_stress(i, t)  = Σ_k [ disruption_prob_k × impact_k × contagion_k(i) ]  ∈ [0,100]
edge_weight(i→j)   = trade_flow (mb/d) × dependency_ratio × substitutability⁻¹  ∈ [0,1]
node_radius        ∝ sector_weight × (1 + stress/100)
layout             = spring_layout seeded by sector cluster centroids
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import networkx as nx
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── Dark terminal aesthetic ──────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  "#07080a",
    "axes.facecolor":    "#0d1117",
    "axes.edgecolor":    "#1c2535",
    "axes.labelcolor":   "#94a3b8",
    "axes.titlecolor":   "#e2e8f0",
    "xtick.color":       "#475569",
    "ytick.color":       "#475569",
    "grid.color":        "#1c2535",
    "text.color":        "#e2e8f0",
    "font.family":       "monospace",
    "figure.dpi":        110,
})

STRESS_CMAP = LinearSegmentedColormap.from_list(
    "stress", ["#4ade80", "#facc15", "#fb923c", "#f43f5e"]
)

def stress_color(s):
    """
    Stress → hex color.
    Pseudocode: piecewise linear map on 4 risk zones [0,30,60,80,100].
    """
    if s < 30:  return "#4ade80"   # green  — LOW
    if s < 60:  return "#facc15"   # yellow — MODERATE
    if s < 80:  return "#fb923c"   # orange — HIGH
    return "#f43f5e"               # red    — CRITICAL

def stress_label(s):
    return {True: "LOW", s<60: "MOD", s<80: "HIGH"}.get(s<30, "CRIT")

print("Libraries loaded ✓")


## 1 · Data — Nodes, Edges, Price Timeline

In [ ]:
# ── NODES ────────────────────────────────────────────────────
# Each node: (id, label, sector, weight, stress_feb, stress_mar)
# weight      = relative sectoral importance (production vol × price elasticity)
# stress_feb  = calibrated to IEA/EIA Feb 2026 data
# stress_mar  = calibrated post-Hormuz closure (Kpler/Wikipedia)

NODES = [
    ("HORMUZ",   "Hormuz Strait",    "TRANSIT",     4.0,  38,  99),
    ("QATAR",    "Qatar LNG",        "UPSTREAM",    1.5,  20,  99),
    ("IRAN",     "Iran",             "UPSTREAM",    1.6,  52,  98),
    ("INSURANCE","Ship Insurance",   "FINANCE",     1.2,  45,  99),
    ("IRAQ",     "Iraq",             "UPSTREAM",    2.0,  28,  88),
    ("JAPAN",    "Japan/S.Korea",    "DOWNSTREAM",  1.2,  25,  88),
    ("OIL_FUT",  "Oil Futures",      "FINANCE",     1.5,  40,  88),
    ("INDIA_REF","India Refining",   "DOWNSTREAM",  1.4,  48,  82),
    ("POWER",    "Power Gen",        "CONSUMER",    1.5,  42,  82),
    ("AVIATION", "Aviation",         "CONSUMER",    1.0,  25,  78),
    ("EU_REF",   "EU Refining",      "DOWNSTREAM",  1.6,  40,  75),
    ("KUWAIT",   "Kuwait",           "UPSTREAM",    1.2,  18,  72),
    ("FERTILIZER","Fertilizers",     "CONSUMER",    1.0,  30,  72),
    ("EQUITIES", "Energy Equities",  "FINANCE",     1.0,  35,  72),
    ("SUEZ",     "Suez/Red Sea",     "TRANSIT",     2.0,  45,  72),
    ("CHINA_REF","China Refining",   "DOWNSTREAM",  2.2,  22,  68),
    ("PETROCHEM","Petrochemicals",   "CONSUMER",    1.3,  28,  65),
    ("SAUDI",    "Saudi Arabia",     "UPSTREAM",    3.0,  18,  62),
    ("RUSSIA",   "Russia",           "UPSTREAM",    2.4,  55,  58),
    ("KAZAKH",   "Kazakhstan",       "UPSTREAM",    1.0,  62,  55),
    ("UAE",      "UAE",              "UPSTREAM",    1.7,  15,  55),
    ("TRANSPORT","Road Transport",   "CONSUMER",    1.8,  20,  55),
    ("MALACCA",  "Malacca Strait",   "TRANSIT",     1.8,  20,  45),
    ("CPC",      "CPC Terminal",     "TRANSIT",     1.0,  65,  55),
    ("OPEC_SP",  "OPEC+ Spare",      "ALT_SUPPLY",  1.5,  25,  45),
    ("IEA_RES",  "IEA Reserves",     "STORAGE",     1.2,  20,  35),
    ("CHINA_SPR","China SPR",        "STORAGE",     1.5,  15,  32),
    ("US_REF",   "US Refining",      "DOWNSTREAM",  1.5,  30,  35),
    ("NORWAY",   "Norway/N.Sea",     "ALT_SUPPLY",  0.8,  22,  30),
    ("US_LNG",   "US LNG Export",    "ALT_SUPPLY",  1.3,  20,  18),
    ("US_PROD",  "US Production",    "UPSTREAM",    2.8,  35,  22),
    ("VENEZUELA","Venezuela",        "UPSTREAM",    0.7,  42,  38),
]

# ── EDGES ────────────────────────────────────────────────────
# (source, target, w_feb, w_mar, flow_type)
# w ∈ [0,1]: trade_flow_mb_d × dependency_ratio × substitutability⁻¹

EDGES = [
    # Upstream → Transit
    ("IRAN",     "HORMUZ",    0.90, 0.05, "supply"),
    ("IRAQ",     "HORMUZ",    0.95, 0.10, "supply"),
    ("KUWAIT",   "HORMUZ",    0.90, 0.15, "supply"),
    ("SAUDI",    "HORMUZ",    0.60, 0.30, "supply"),
    ("QATAR",    "HORMUZ",    0.85, 0.01, "supply"),
    ("UAE",      "HORMUZ",    0.55, 0.25, "supply"),
    ("RUSSIA",   "CPC",       0.60, 0.50, "supply"),
    ("KAZAKH",   "CPC",       0.80, 0.70, "supply"),
    # Transit → Downstream
    ("HORMUZ",   "JAPAN",     0.75, 0.08, "transport"),
    ("HORMUZ",   "INDIA_REF", 0.65, 0.10, "transport"),
    ("HORMUZ",   "CHINA_REF", 0.55, 0.15, "transport"),
    ("SUEZ",     "EU_REF",    0.55, 0.40, "transport"),
    ("SUEZ",     "US_REF",    0.30, 0.25, "transport"),
    ("CPC",      "EU_REF",    0.45, 0.40, "transport"),
    ("MALACCA",  "CHINA_REF", 0.40, 0.50, "transport"),
    ("MALACCA",  "JAPAN",     0.45, 0.55, "transport"),
    # Downstream → Consumer
    ("EU_REF",   "POWER",     0.50, 0.75, "consume"),
    ("EU_REF",   "FERTILIZER",0.40, 0.70, "consume"),
    ("EU_REF",   "AVIATION",  0.50, 0.35, "consume"),
    ("EU_REF",   "TRANSPORT", 0.60, 0.55, "consume"),
    ("INDIA_REF","AVIATION",  0.45, 0.20, "consume"),
    ("INDIA_REF","PETROCHEM", 0.35, 0.25, "consume"),
    ("CHINA_REF","PETROCHEM", 0.55, 0.40, "consume"),
    ("JAPAN",    "POWER",     0.45, 0.30, "consume"),
    ("US_REF",   "TRANSPORT", 0.60, 0.65, "consume"),
    # Finance
    ("HORMUZ",   "INSURANCE", 0.55, 0.99, "finance"),
    ("INSURANCE","OIL_FUT",   0.40, 0.85, "finance"),
    ("OIL_FUT",  "EQUITIES",  0.55, 0.80, "finance"),
    ("IRAN",     "OIL_FUT",   0.40, 0.90, "finance"),
    # Mitigation
    ("US_LNG",   "EU_REF",    0.25, 0.55, "mitigate"),
    ("NORWAY",   "EU_REF",    0.30, 0.45, "mitigate"),
    ("IEA_RES",  "EU_REF",    0.20, 0.55, "mitigate"),
    ("IEA_RES",  "JAPAN",     0.20, 0.60, "mitigate"),
    ("CHINA_SPR","CHINA_REF", 0.15, 0.50, "mitigate"),
    ("US_PROD",  "US_LNG",    0.50, 0.70, "supply"),
    ("RUSSIA",   "CHINA_REF", 0.70, 0.75, "supply"),
    ("VENEZUELA","INDIA_REF", 0.30, 0.35, "supply"),
    ("OPEC_SP",  "HORMUZ",    0.30, 0.20, "mitigate"),
]

PRICE_DATA = [
    ("Dec'25",62,None), ("Jan 1",62,None),
    ("Jan 15",67,"Winter Storm + Kazakhstan"),
    ("Jan 31",72,None), ("Feb 1",70,None),
    ("Feb 10",67,"EIA: oversupply bearish"),
    ("Feb 15",67,"Iran military drill"),
    ("Feb 20",71,None),
    ("Feb 28",73,"Operation Epic Fury"),
    ("Mar 1",80,"Hormuz de-facto closed"),
    ("Mar 2",83,"Qatar halts LNG"),
    ("Mar 3",88,"Peak — 300 tankers trapped"),
    ("Mar 4",85,"Qatar Force Majeure"),
    ("Mar 5",83,"P&I insurance withdrawn"),
    ("Mar 6",83,"Current level"),
]

# Build DataFrames
df_nodes = pd.DataFrame(NODES, columns=["id","label","sector","weight","s_feb","s_mar"])
df_nodes["delta"]   = df_nodes["s_mar"] - df_nodes["s_feb"]
df_nodes["col_feb"] = df_nodes["s_feb"].apply(stress_color)
df_nodes["col_mar"] = df_nodes["s_mar"].apply(stress_color)

df_edges = pd.DataFrame(EDGES, columns=["source","target","w_feb","w_mar","flow_type"])
df_edges["delta"] = df_edges["w_mar"] - df_edges["w_feb"]

df_price = pd.DataFrame(PRICE_DATA, columns=["date","brent","event"])

df_sector = (df_nodes.groupby("sector")
             .agg(mean_feb=("s_feb","mean"), mean_mar=("s_mar","mean"))
             .reset_index())
df_sector["delta"] = df_sector["mean_mar"] - df_sector["mean_feb"]

print(f"Nodes: {len(df_nodes)}  |  Edges: {len(df_edges)}  |  Price points: {len(df_price)}")
df_nodes[["label","sector","s_feb","s_mar","delta"]].sort_values("delta", ascending=False).head(10)


## 2 · Build Directed Weighted Graphs

In [ ]:
SECTOR_COLORS = {
    "UPSTREAM":   "#4FC3F7",
    "TRANSIT":    "#FF8A65",
    "DOWNSTREAM": "#81C784",
    "CONSUMER":   "#CE93D8",
    "FINANCE":    "#FFD54F",
    "ALT_SUPPLY": "#80CBC4",
    "STORAGE":    "#90A4AE",
}

FLOW_COLORS = {
    "supply":    "#4FC3F7",
    "transport": "#FF8A65",
    "consume":   "#81C784",
    "finance":   "#FFD54F",
    "mitigate":  "#80CBC4",
}

# Cluster seed positions: sector centroids in layout space
# Each sector gets a (x,y) anchor; nodes start near anchor + Gaussian noise
# Then spring_layout relaxes toward minimum energy configuration
CLUSTER_POS = {
    "UPSTREAM":   np.array([ 0.0,  0.6]),
    "TRANSIT":    np.array([ 0.0,  0.0]),
    "DOWNSTREAM": np.array([ 0.0, -0.6]),
    "CONSUMER":   np.array([ 0.9, -0.3]),
    "FINANCE":    np.array([ 0.9,  0.3]),
    "ALT_SUPPLY": np.array([-0.9, -0.3]),
    "STORAGE":    np.array([-0.9,  0.3]),
}

def build_graph(time_key):
    """
    Pseudocode:
      G = DiGraph()
      for each node n: G.add_node(n, stress=s[n,t], weight=w[n])
      for each edge (i,j): if w(i,j,t) > threshold: G.add_edge(i,j, weight=w)
      return G
    """
    G = nx.DiGraph()
    s_col = f"s_{time_key}"
    w_col = f"w_{time_key}"
    for _, n in df_nodes.iterrows():
        G.add_node(n["id"], label=n["label"], sector=n["sector"],
                   weight=n["weight"], stress=n[s_col], delta=n["delta"])
    for _, e in df_edges.iterrows():
        w = e[w_col]
        if w > 0.02:
            G.add_edge(e["source"], e["target"],
                       weight=w, flow_type=e["flow_type"])
    return G

def cluster_layout(G, seed=7):
    """
    Pseudocode:
      pos0 = {n: cluster_center[sector(n)] + Gaussian(0, 0.18)}
      pos  = spring_layout(G, pos0, k=0.55, iterations=80)
      # k controls equilibrium distance; iterations = relaxation steps
    """
    np.random.seed(seed)
    pos0 = {nid: CLUSTER_POS[data["sector"]] + np.random.randn(2)*0.18
            for nid, data in G.nodes(data=True)}
    return nx.spring_layout(G, pos=pos0, k=0.55, iterations=80,
                            weight="weight", seed=seed)

G_feb = build_graph("feb")
G_mar = build_graph("mar")
pos_feb = cluster_layout(G_feb)
pos_mar = cluster_layout(G_mar)

print(f"Feb graph: {G_feb.number_of_nodes()} nodes, {G_feb.number_of_edges()} edges")
print(f"Mar graph: {G_mar.number_of_nodes()} nodes, {G_mar.number_of_edges()} edges")


## 3 · Figure 1 — Supply Chain Network Comparison

In [ ]:
def draw_network(ax, G, pos, title, time_key, show_legend=False):
    ax.set_facecolor("#0d1117")
    ax.set_title(title, color="#e2e8f0", fontsize=10, fontweight="bold",
                 pad=10, fontfamily="monospace")
    ax.axis("off")
    s_col = f"s_{time_key}"

    # Draw edges — width ∝ flow weight
    for u, v, data in G.edges(data=True):
        w = data["weight"]
        fc = FLOW_COLORS.get(data["flow_type"], "#888")
        style = "--" if data["flow_type"] == "mitigate" else "-"
        nx.draw_networkx_edges(G, pos, edgelist=[(u,v)],
                               width=w*3.5, alpha=min(0.85, 0.2+w*0.7),
                               edge_color=fc, style=style,
                               arrows=True, arrowsize=10,
                               connectionstyle="arc3,rad=0.08", ax=ax)

    # Draw nodes — radius ∝ weight, color ∝ stress
    for nid, ndata in G.nodes(data=True):
        s = ndata["stress"]
        r = ndata["weight"] * 200 + s * 3
        col = stress_color(s)
        if s >= 80:   # glow halo for critical nodes
            nx.draw_networkx_nodes(G, pos, nodelist=[nid],
                                   node_size=r*2.5, node_color=col,
                                   alpha=0.12, ax=ax)
        nx.draw_networkx_nodes(G, pos, nodelist=[nid],
                               node_size=r, node_color=col, alpha=0.82,
                               linewidths=1.5 if s>=60 else 0.6,
                               edgecolors=col, ax=ax)

    labels = {n: G.nodes[n]["label"] for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=6.2,
                            font_color="#cbd5e1", font_family="monospace", ax=ax)

    # Stress index badges
    for nid, (x, y) in pos.items():
        s = G.nodes[nid]["stress"]
        ax.text(x, y+0.06, str(s), fontsize=5.5, color=stress_color(s),
                ha="center", va="bottom", fontweight="bold", fontfamily="monospace")

    if show_legend:
        handles = [mpatches.Patch(color=v, label=k.replace("_"," "))
                   for k,v in SECTOR_COLORS.items()]
        ax.legend(handles=handles, loc="lower left", fontsize=6,
                  framealpha=0.3, facecolor="#0d1117",
                  edgecolor="#1c2535", labelcolor="#94a3b8")

fig1, axes = plt.subplots(1, 2, figsize=(18, 9))
fig1.patch.set_facecolor("#07080a")
fig1.suptitle(
    "OIL SUPPLY CHAIN STRESS NETWORK  ·  Node radius ∝ weight  "
    "·  Node color ∝ stress  ·  Edge width ∝ dependency",
    color="#64748b", fontsize=8, fontfamily="monospace"
)

draw_network(axes[0], G_feb, pos_feb,
             "FEB 15, 2026 — Distributed Stress (Brent $67/bbl)", "feb",
             show_legend=True)
draw_network(axes[1], G_mar, pos_mar,
             "MAR 6, 2026 — Hub-Collapse Regime (Brent $83/bbl, Hormuz closed)", "mar")

stress_handles = [
    mpatches.Patch(color="#4ade80", label="LOW < 30"),
    mpatches.Patch(color="#facc15", label="MOD 30–60"),
    mpatches.Patch(color="#fb923c", label="HIGH 60–80"),
    mpatches.Patch(color="#f43f5e", label="CRIT > 80"),
    Line2D([0],[0], color="#80CBC4", lw=1.5, ls="--", label="Mitigating flow"),
]
fig1.legend(handles=stress_handles, loc="lower center", ncol=5,
            fontsize=7.5, framealpha=0.4, facecolor="#0d1117",
            edgecolor="#1c2535", labelcolor="#94a3b8")

plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()


## 4 · Figure 2 — Analytics Dashboard

In [ ]:
fig2 = plt.figure(figsize=(20, 16))
fig2.patch.set_facecolor("#07080a")
gs = gridspec.GridSpec(3, 2, figure=fig2, hspace=0.52, wspace=0.32,
                       left=0.06, right=0.97, top=0.94, bottom=0.06)

ax_bar   = fig2.add_subplot(gs[0, :])
ax_delta = fig2.add_subplot(gs[1, 0])
ax_sect  = fig2.add_subplot(gs[1, 1])
ax_price = fig2.add_subplot(gs[2, 0])
ax_heat  = fig2.add_subplot(gs[2, 1])

fig2.text(0.5, 0.97,
    "OIL SUPPLY CHAIN  ·  STRESS ANALYTICS DASHBOARD  ·  FEB 15 vs MAR 6, 2026",
    ha="center", color="#e2e8f0", fontsize=12, fontfamily="monospace", fontweight="bold")

# ── #01 Grouped bar ───────────────────────────────────────────
ax = ax_bar
sn = df_nodes.sort_values("s_mar", ascending=False)
x = np.arange(len(sn)); bw = 0.38
ax.bar(x-bw/2, sn["s_feb"], bw, color="#38bdf8", alpha=0.75, label="Feb 15", zorder=3)
ax.bar(x+bw/2, sn["s_mar"], bw, color=[stress_color(s) for s in sn["s_mar"]],
       alpha=0.88, label="Mar 6", zorder=3)
for lo,hi,c in [(0,30,"#4ade80"),(30,60,"#facc15"),(60,80,"#fb923c"),(80,100,"#f43f5e")]:
    ax.axhspan(lo, hi, alpha=0.04, color=c, zorder=1)
ax.set_xticks(x)
ax.set_xticklabels(sn["label"], rotation=40, ha="right",
                   fontsize=7.5, fontfamily="monospace")
ax.set_ylabel("Stress Index (0–100)"); ax.set_ylim(0,108)
ax.set_title("#01 — NODE STRESS INDEX: FEB 15 vs MAR 6",
             color="#94a3b8", fontsize=9, fontfamily="monospace")
ax.legend(loc="upper right", fontsize=8, framealpha=0.3,
          facecolor="#0d1117", edgecolor="#1c2535", labelcolor="#94a3b8")
ax.grid(axis="y", alpha=0.3)
for sp in ax.spines.values(): sp.set_edgecolor("#1c2535")

# ── #02 Delta lollipop ────────────────────────────────────────
ax = ax_delta
sd = df_nodes.sort_values("delta", ascending=True)
y = np.arange(len(sd))
for i, (_, row) in enumerate(sd.iterrows()):
    col = stress_color(row["s_mar"]) if row["delta"] >= 0 else "#4ade80"
    ax.plot([0, row["delta"]], [i, i], color=col, alpha=0.7, lw=1.5)
    ax.scatter(row["delta"], i, color=col, s=30+abs(row["delta"])*1.5,
               zorder=3, edgecolors="#07080a", linewidths=0.8)
ax.axvline(0, color="#334155", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(sd["label"], fontsize=7, fontfamily="monospace")
ax.set_xlabel("Δ Stress (Mar − Feb)", fontsize=8)
ax.set_title("#02 — Δ STRESS SHIFT (MAR − FEB)",
             color="#94a3b8", fontsize=9, fontfamily="monospace")
ax.grid(axis="x", alpha=0.3)
for lbl, (_, row) in zip(ax.get_yticklabels(), sd.iterrows()):
    lbl.set_color(stress_color(row["s_mar"]))
for sp in ax.spines.values(): sp.set_edgecolor("#1c2535")

# ── #03 Sector stress ─────────────────────────────────────────
ax = ax_sect
ss = df_sector.sort_values("mean_mar", ascending=False)
xs = np.arange(len(ss))
ax.bar(xs-0.2, ss["mean_feb"], 0.38, color="#38bdf8", alpha=0.75, label="Feb 15")
ax.bar(xs+0.2, ss["mean_mar"], 0.38,
       color=[stress_color(s) for s in ss["mean_mar"]], alpha=0.88, label="Mar 6")
ax.set_xticks(xs)
ax.set_xticklabels([s[:9] for s in ss["sector"]],
                   rotation=30, ha="right", fontsize=8, fontfamily="monospace")
ax.set_ylabel("Mean Stress Index"); ax.set_ylim(0,110)
ax.set_title("#03 — SECTOR MEAN STRESS",
             color="#94a3b8", fontsize=9, fontfamily="monospace")
ax.legend(fontsize=8, framealpha=0.3, facecolor="#0d1117",
          edgecolor="#1c2535", labelcolor="#94a3b8")
ax.grid(axis="y", alpha=0.3)
for i,(_, row) in enumerate(ss.iterrows()):
    d = row["mean_mar"] - row["mean_feb"]
    ax.text(i+0.2, row["mean_mar"]+2, f"+{d:.0f}" if d>0 else f"{d:.0f}",
            ha="center", fontsize=7, color=stress_color(row["mean_mar"]),
            fontfamily="monospace", fontweight="bold")
for sp in ax.spines.values(): sp.set_edgecolor("#1c2535")

# ── #04 Brent price timeline ──────────────────────────────────
ax = ax_price
dates = df_price["date"].tolist(); prices = df_price["brent"].tolist()
xi = np.arange(len(dates))
cs = dates.index("Feb 28")
ax.axvspan(cs, len(dates)-1, alpha=0.07, color="#fb923c")
ax.text(cs+0.2, 91, "← CONFLICT", color="#fb923c", alpha=0.5,
        fontsize=7.5, fontfamily="monospace")
ax.plot(xi, prices, color="#38bdf8", lw=2.5)
ax.plot(xi[cs:], prices[cs:], color="#fb923c", lw=2.5)
ax.fill_between(xi, prices, min(prices)-2, color="#38bdf8", alpha=0.06)
ax.axhline(67, color="#38bdf8", lw=1, ls="--", alpha=0.35)
ax.axhline(83, color="#fb923c", lw=1, ls="--", alpha=0.35)
ax.text(len(dates)-0.5, 67.5, "$67", color="#38bdf8",
        fontsize=7.5, ha="right", fontfamily="monospace")
ax.text(len(dates)-0.5, 83.5, "$83", color="#fb923c",
        fontsize=7.5, ha="right", fontfamily="monospace")
for i, row in df_price.iterrows():
    if row["event"]:
        col = "#f43f5e" if i>=cs else "#facc15"
        ax.scatter(i, row["brent"], color=col, s=45, zorder=5,
                   edgecolors="#07080a", linewidths=1)
        yoff = 1.5 if i%2==0 else -3.5
        ax.text(i, row["brent"]+yoff, row["event"],
                color=col, fontsize=5.5, ha="center",
                fontfamily="monospace", alpha=0.8)
ax.set_xticks(xi)
ax.set_xticklabels(dates, rotation=40, ha="right",
                   fontsize=7, fontfamily="monospace")
ax.set_ylabel("Brent ($/bbl)"); ax.set_ylim(56, 96)
ax.set_title("#04 — BRENT CRUDE PRICE TRAJECTORY",
             color="#94a3b8", fontsize=9, fontfamily="monospace")
ax.grid(alpha=0.3)
for sp in ax.spines.values(): sp.set_edgecolor("#1c2535")

# ── #05 Flow heatmap ──────────────────────────────────────────
ax = ax_heat
hd = df_edges.copy()
hd["label"] = hd["source"].str[:5]+"→"+hd["target"].str[:5]
hm = hd[["w_feb","w_mar","delta"]].values.T
im = ax.imshow(hm, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
ax.set_yticks([0,1,2])
ax.set_yticklabels(["Feb Weight","Mar Weight","Δ Change"],
                   fontsize=8, fontfamily="monospace")
ax.set_xticks(np.arange(len(hd)))
ax.set_xticklabels(hd["label"], rotation=55, ha="right",
                   fontsize=6, fontfamily="monospace")
for ci in range(hm.shape[0]):
    for cj in range(hm.shape[1]):
        val = hm[ci, cj]
        txt = f"{val:+.2f}" if ci==2 else f"{val:.2f}"
        ax.text(cj, ci, txt, ha="center", va="center", fontsize=5.5,
                color="white" if abs(val)>0.55 else "#94a3b8",
                fontfamily="monospace",
                fontweight="bold" if abs(val)>0.4 else "normal")
plt.colorbar(im, ax=ax, shrink=0.6, label="Weight / Δ")
ax.set_title("#05 — DEPENDENCY FLOW HEATMAP",
             color="#94a3b8", fontsize=9, fontfamily="monospace")

plt.show()


## 5 · Network Centrality Metrics

In [ ]:
def network_metrics(G):
    """
    Compute graph-theoretic observables.

    Betweenness centrality BC(v):
      BC(v) = Σ_{s≠v≠t} σ(s,t|v) / σ(s,t)
      where σ(s,t) = number of shortest paths from s to t
      High BC → bottleneck node; removal causes maximum network fragmentation

    PageRank PR(v):
      PR(v) = (1-α)/N + α × Σ_{u→v} PR(u)/out_degree(u)
      Stationary distribution of weighted random walk; α=0.85 (damping)
      High PR → node receives most 'flow' from the rest of the network

    Clustering coefficient CC(v):
      CC(v) = (triangles through v) / (possible triangles)
      High CC → node embedded in tightly coupled cluster (fragile to cascade)
    """
    bc = nx.betweenness_centrality(G, weight="weight", normalized=True)
    dc = nx.degree_centrality(G)
    pr = nx.pagerank(G, weight="weight", alpha=0.85)
    cc = nx.clustering(G.to_undirected(), weight="weight")
    return {n: {"BC": bc[n], "DC": dc[n], "PR": pr[n], "CC": cc[n]}
            for n in G.nodes()}

mf = network_metrics(G_feb)
mm = network_metrics(G_mar)

rows = []
for _, n in df_nodes.iterrows():
    nid = n["id"]
    if nid not in mf: continue
    rows.append({
        "Node": n["label"], "Sector": n["sector"],
        "s_feb": n["s_feb"], "s_mar": n["s_mar"], "Δ": n["delta"],
        "BC_feb": round(mf[nid]["BC"],4), "BC_mar": round(mm[nid]["BC"],4),
        "PR_feb": round(mf[nid]["PR"],4), "PR_mar": round(mm[nid]["PR"],4),
        "CC_feb": round(mf[nid]["CC"],3), "CC_mar": round(mm[nid]["CC"],3),
    })

df_metrics = pd.DataFrame(rows).sort_values("Δ", ascending=False)

# Plot
fig3, ax = plt.subplots(figsize=(18, 10))
fig3.patch.set_facecolor("#07080a"); ax.axis("off")
ax.set_title("NETWORK CENTRALITY METRICS — Top 20 by Δ Stress",
             color="#e2e8f0", fontsize=10, fontfamily="monospace",
             fontweight="bold", pad=12)

top = df_metrics.head(20)
cols = ["Node","s_feb","s_mar","Δ","BC_feb","BC_mar","PR_feb","PR_mar","CC_feb","CC_mar"]
hdrs = ["Node","Feb","Mar","Δ","BC Feb","BC Mar","PR Feb","PR Mar","CC Feb","CC Mar"]

tbl = ax.table(cellText=top[cols].values.tolist(),
               colLabels=hdrs, cellLoc="center", loc="center",
               bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(8)

for (r,c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#1c2535"); cell.set_linewidth(0.5)
    if r == 0:
        cell.set_facecolor("#1c2535")
        cell.set_text_props(color="#94a3b8", fontfamily="monospace",
                            fontweight="bold", fontsize=7.5)
    else:
        row_d = top.iloc[r-1]
        cell.set_facecolor("#0d1117")
        if   c == 0: cell.set_text_props(color="#e2e8f0",  fontfamily="monospace")
        elif c == 1: cell.set_text_props(color="#38bdf8",  fontfamily="monospace")
        elif c == 2: cell.set_text_props(color=stress_color(row_d["s_mar"]),
                                         fontfamily="monospace", fontweight="bold")
        elif c == 3:
            d = row_d["Δ"]
            cell.set_text_props(color=stress_color(row_d["s_mar"]) if d>0 else "#4ade80",
                                fontfamily="monospace", fontweight="bold")
        else:        cell.set_text_props(color="#64748b",  fontfamily="monospace")

plt.show()
print(df_metrics[["Node","s_feb","s_mar","Δ","BC_feb","BC_mar"]].to_string(index=False))


## 6 · Summary Statistics

In [ ]:
mean_feb = df_nodes["s_feb"].mean()
mean_mar = df_nodes["s_mar"].mean()
crit_feb = (df_nodes["s_feb"]>=80).sum()
crit_mar = (df_nodes["s_mar"]>=80).sum()

print("═"*60)
print("  PHASE TRANSITION STATISTICS")
print("═"*60)
print(f"  Mean network stress : {mean_feb:.1f} → {mean_mar:.1f}  "
      f"(+{mean_mar-mean_feb:.1f}, +{(mean_mar-mean_feb)/mean_feb*100:.0f}%)")
print(f"  Critical nodes ≥80  : {crit_feb} → {crit_mar}")
print(f"  Brent crude         : $67 → $83/bbl  (+24%)")
print()
print("  TOP COLLAPSED FLOWS")
for _, e in df_edges.sort_values("delta").head(5).iterrows():
    print(f"    {e['source']:10s} → {e['target']:10s}  "
          f"{e['w_feb']:.2f} → {e['w_mar']:.2f}  Δ={e['delta']:+.2f}")
print()
print("  TOP SURGED FLOWS")
for _, e in df_edges.sort_values("delta",ascending=False).head(5).iterrows():
    print(f"    {e['source']:10s} → {e['target']:10s}  "
          f"{e['w_feb']:.2f} → {e['w_mar']:.2f}  Δ={e['delta']:+.2f}")
print()
print("  SECTOR STRESS AMPLIFICATION")
for _, row in df_sector.sort_values("delta",ascending=False).iterrows():
    bar = "█" * int(row["delta"]/3)
    print(f"    {row['sector']:14s}  Δ=+{row['delta']:.1f}  {bar}")
print("═"*60)
